In [3]:
# 可视化CT数据实现

# simpleITK ipyvolume diskcache cassandra-driver

%pip install simpleITK ipyvolume diskcache cassandra-driver


Looking in indexes: http://mirrors.aliyun.com/pypi/simpleNote: you may need to restart the kernel to use updated packages.

     ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
     - -------------------------------------- 0.5/18.7 MB 4.2 MB/s eta 0:00:05
     -- ------------------------------------- 1.3/18.7 MB 3.7 MB/s eta 0:00:05
     ---- ----------------------------------- 2.1/18.7 MB 3.8 MB/s eta 0:00:05
     ------ --------------------------------- 2.9/18.7 MB 3.7 MB/s eta 0:00:05
     -------- ------------------------------- 3.9/18.7 MB 3.8 MB/s eta 0:00:04
     ---------- ----------------------------- 4.7/18.7 MB 3.8 MB/s eta 0:00:04
     ----------- ---------------------------- 5.5/18.7 MB 3.8 MB/s eta 0:00:04
     ------------- -------------------------- 6.3/18.7 MB 3.8 MB/s eta 0:00:04
     --------------- ------------------------ 7.1/18.7 MB 3.8 MB/s eta 0:00:04
     ---------------- ----------------------- 7.9/18.7 MB 3.8 MB/s eta 0:00:03
     ---------

In [13]:
import gzip
from diskcache import FanoutCache, Disk
from cassandra.cqltypes import BytesType
from diskcache import FanoutCache, Disk, core
from diskcache.core import io
from io import BytesIO
from diskcache.core import  MODE_BINARY
from util.logconf import logging
log = logging.getLogger(__name__)
log.setLevel(logging.INFO)

import matplotlib
matplotlib.use('nbagg')

import numpy as np
import matplotlib.pyplot as plt
import sys

from code1.dsets import Ct, LunaDataset

clim = (-1000.0, 300)

In [14]:
def findPositiveSamples(start_ndx=0, limit=100):
    ds = LunaDataset()
    positiveSample_list = []
    for sample_tup in ds.candidateInfo_list:
        if sample_tup.isNodule_bool:
            print(len(positiveSample_list), sample_tup)
            positiveSample_list.append(sample_tup)
        if len(positiveSample_list) >= limit:
            break
    return positiveSample_list

In [16]:
def showCandidate(series_uid, batch_ndx=None, **kwargs):
    ds = LunaDataset(series_uid=series_uid, **kwargs)
    pos_list = [i for i , x in enumerate(ds.candidateInfo_list) if x.isNodule_bool]
    
    if batch_ndx is None:
        if pos_list:
            batch_ndx = pos_list[0]
        else:
            print("Warning : no positive samples found; using first negative sample.")
            batch_ndx = 0
            
    ct = Ct(series_uid)
    ct_t, pos_t, series_uid, center_irc = ds[batch_ndx]
    ct_a = ct_t[0].numpy()
    
    fig = plt.figure(figsize=(30, 50))
    
    group_list = [[9, 11, 13],[15, 16, 17],[19, 21, 23]]
    
    #add_subplot(3, 4, 9)
    subplot = fig.add_subplot(len(group_list) + 2, 3, 1)
    subplot.set_title('index {}'.format(int(center_irc[0])), fontsize=30)
    for label in (subplot.get_xticklabels() + subplot.get_yticklabels()):
        label.set_fontsize(20)
    plt.imshow(ct.hu_a[int(center_irc[0])], clim=clim, cmap='gray')
    
    
    subplot = fig.add_subplot(len(group_list) + 2, 3, 2)
    subplot.set_title('row {}'.format(int(center_irc[1])), fontsize=30)
    for label in (subplot.get_xticklabels() + subplot.get_yticklabels()):
        label.set_fontsize(20)
    plt.imshow(ct.hu_a[:,int(center_irc[1])], clim=clim, cmap='gray')#取行信息展示，对应我们的正视图
    plt.gca().invert_yaxis()

    subplot = fig.add_subplot(len(group_list) + 2, 3, 3)
    subplot.set_title('col {}'.format(int(center_irc[2])), fontsize=30)
    for label in (subplot.get_xticklabels() + subplot.get_yticklabels()):
        label.set_fontsize(20)
    plt. imshow(ct.hu_a[:,:,int(center_irc[2])], clim=clim, cmap='gray')#取列信息展示，对应我们的侧视图
    plt.gca().invert_yaxis()

    subplot = fig.add_subplot(len(group_list) + 2, 3, 4)
    subplot.set_title('index {}'.format(int(center_irc[0])), fontsize=30)
    for label in (subplot.get_xticklabels() + subplot.get_yticklabels()):
        label.set_fontsize(20)
    plt.imshow(ct_a[ct_a.shape[0]//2], clim=clim, cmap='gray')#这个是取候选小块的信息，除2是放大？

    subplot = fig.add_subplot(len(group_list) + 2, 3, 5)
    subplot.set_title('row {}'.format(int(center_irc[1])), fontsize=30)
    for label in (subplot.get_xticklabels() + subplot.get_yticklabels()):
        label.set_fontsize(20)
    plt.imshow(ct_a[:,ct_a.shape[1]//2], clim=clim, cmap='gray')#候选小块的行信息
    plt.gca().invert_yaxis()

    subplot = fig.add_subplot(len(group_list) + 2, 3, 6)
    subplot.set_title('col {}'.format(int(center_irc[2])), fontsize=30)
    for label in (subplot.get_xticklabels() + subplot.get_yticklabels()):
        label.set_fontsize(20)
    plt.imshow(ct_a[:,:,ct_a.shape[2]//2], clim=clim, cmap='gray') #候选小块的列信息
    plt.gca().invert_yaxis()
    
    
    for row, index_list in enumerate(group_list):
        for col, index in enumerate(index_list):
            subplot = fig.add_subplot(len(group_list) + 2, 3, row * 3 + col + 7)
            subplot.set_title('slice {}'.format(index), fontsize=30)
            for label in (subplot.get_xticklabels() + subplot.get_yticklabels()):
                label.set_fontsize(20)
            plt.imshow(ct_a[index], clim=clim, cmap='gary')
    
    print(series_uid, batch_ndx, bool(pos_t[0]), pos_list)

In [17]:
%matplotlib inline
from code1.vis import findPositiveSamples, showCandidate
positiveSample_list = findPositiveSamples()

FileNotFoundError: [Errno 2] No such file or directory: 'D:/pytorchProject/data/lunadata/annotations.csv'